In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph, MessagesState
from langchain_core.tools import tool
from typing import Literal
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv()

True

In [ ]:
embeddings = OpenAIEmbeddings()
parser = StrOutputParser()

In [8]:
class Chatbot:
    def __init__(self,system=""):
        self.system=system
        self.message=[]
        if self.system:
            self.message.append({"role":"system","content":system})
            
    def __call__(self,message):
        self.message.append({"role":"user","content":message})
        result=self.execute()
        self.message.append({"role":"assistant","content":result})
        return result
        
    def execute(self):
        llm = ChatOpenAI(model='gpt-4o-mini')
        result = llm.invoke(self.message)
        return result.content

In [9]:
bot = Chatbot("you are a helpful assistant")
bot("Hi! How are you?")

"Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"

In [10]:
bot.message

[{'role': 'system', 'content': 'you are a helpful assistant'},
 {'role': 'user', 'content': 'Hi! How are you?'},
 {'role': 'assistant',
  'content': "Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"}]

In [11]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop your output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.


Your available actions are:
calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point
syntax if necessary

wikipedia:
e.g. wikipedia: Django
Returns a summary from searching Wikipedia

simon_blog_search:
e.g. simon_blog_search: Python
Search Simon's blog for that term

Example session:
Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:
Observation: France is a country. The capital is Paris.

You then output:
Answer: The capital of France is Paris

Please Note: if you get basic conversation questions like "hi","hello","how are you?",\n
you have to answer "hi","hello","i am good".
""".strip()

In [13]:
import re
action_re = re.compile('^Action: (\w+): (.*)')

In [12]:
text1 = "Action: Move: North"
text2 = "Action: Jump: High"
text3 = "Error: Not an Action"

In [14]:
match1 = action_re.match(text1)
match1

<re.Match object; span=(0, 19), match='Action: Move: North'>

In [15]:
match1.group(1)

'Move'

In [16]:
match1.group(2)

'North'

In [17]:
match2 = action_re.match(text2)
match2

<re.Match object; span=(0, 18), match='Action: Jump: High'>

In [18]:
match3 = action_re.match(text3)
match3

In [19]:
# Extracting the matched groups
if match1:
    print(f"Matched: {match1.group(1)}, {match1.group(2)}")

Matched: Move, North


In [20]:
if match2:
    print(f"Matched: {match2.group(1)}, {match2.group(2)}")

Matched: Jump, High


In [21]:
if match3:
    print("No match")  # This won't match, as it doesn't start with "Action:"

In [22]:
import httpx
def wikipedia(query):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    })
    return response.json()["query"]["search"][0]["snippet"]

In [23]:
import httpx
def simon_blog_search(query):
    response = httpx.get("https://datasette.simonwillison.net/simonwillisonblog.json", params={
        "sql": """
        select
          blog_entry.title || ': ' || substr(html_strip_tags(blog_entry.body), 0, 1000) as text,
          blog_entry.created
        from
          blog_entry join blog_entry_fts on blog_entry.rowid = blog_entry_fts.rowid
        where
          blog_entry_fts match escape_fts(:q)
        order by
          blog_entry_fts.rank
        limit
          1
        """.strip(),
        "_shape": "array",
        "q": query,
    })
    return response.json()[0]["text"]

In [24]:
def calculate(number):
    return eval(number)

In [25]:
calculate("2+2")

4

In [26]:
known_actions = {
    "wikipedia": wikipedia,
    "calculate": calculate,
    "simon_blog_search": simon_blog_search
}

In [27]:
bot = Chatbot(prompt)

In [28]:
next_prompt="Tell me about quantum computing from Wikipedia."
result = bot(next_prompt)
result

'Thought: I should look up quantum computing on Wikipedia to get a comprehensive overview of the topic.  \nAction: wikipedia: quantum computing  \nPAUSE'

In [29]:
actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
actions

[<re.Match object; span=(0, 38), match='Action: wikipedia: quantum computing  '>]

In [30]:
action, action_input = actions[0].groups()

In [31]:
action_input

'quantum computing  '

In [32]:
action

'wikipedia'

In [33]:
known_actions[action]

<function __main__.wikipedia(query)>

In [34]:
known_actions[action](action_input)

'and <span class="searchmatch">quantum</span> <span class="searchmatch">computing</span> takes advantage of this behavior using specialized hardware. Classical physics cannot explain the operation of these <span class="searchmatch">quantum</span> devices'

In [35]:
def query(question,max_turns=5):
    i = 0
    bot = Chatbot(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            return result

In [36]:
query("hello")

hi


'hi'

In [37]:
query("how are you?")

hi, hello, i am good.


'hi, hello, i am good.'

In [38]:
query("what you doing?")

hi


'hi'

In [39]:
print(query("Fifteen * twenty five"))

Thought: I need to calculate the product of fifteen and twenty five. 
Action: calculate: 15 * 25
PAUSE
 -- running calculate 15 * 25
Observation: 375
Answer: Fifteen multiplied by twenty five is 375.
Answer: Fifteen multiplied by twenty five is 375.


In [40]:
query("Has Simon written about AI?")

Thought: I should search Simon's blog for articles related to AI to find out if he has written about it.  
Action: simon_blog_search: AI  
PAUSE
 -- running simon_blog_search AI  
Observation: It's OK to call it Artificial Intelligence: Update 9th January 2024: This post was clumsily written and failed to make the point I wanted it to make. I've published a follow-up, What I should have said about the term Artificial Intelligence which you should read instead.
My original post follows.


We need to be having high quality conversations about AI: what it can and can't do, its many risks and pitfalls and how to integrate it into society in the most beneficial ways possible.
Any time I write anything that mentions AI it's inevitable that someone will object to the very usage of the term.
Strawman: "Don't call it AI! It's not actually intelligent - it's just spicy autocomplete."
That strawman is right: it's not "intelligent" in the same way that humans are. And "spicy autocomplete" is actua

'Answer: Yes, Simon has written about AI, discussing the importance of high-quality conversations regarding its capabilities, risks, and integration into society. He also addresses the criticism surrounding the term "Artificial Intelligence."'

In [41]:
query("Langchain")

Thought: I should look up Langchain on Wikipedia to understand what it is and its significance.  
Action: wikipedia: Langchain  
PAUSE
 -- running wikipedia Langchain  
Observation: Free and open-source software portal <span class="searchmatch">LangChain</span> is a software framework that helps facilitate the integration of large language models (LLMs) into applications
Answer: LangChain is a free and open-source software framework that helps facilitate the integration of large language models (LLMs) into applications.


'Answer: LangChain is a free and open-source software framework that helps facilitate the integration of large language models (LLMs) into applications.'